In [ ]:
!pip install torch

In [ ]:
!pip install pytorch_tabnet

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from pytorch_tabnet.tab_model import TabNetClassifier
import wandb

In [ ]:
wandb.login('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')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [ ]:
df = pd.read_csv("/content/cell2celltrain.csv")
y = df["Churn"].map({"Yes": 1, "No": 0}).astype(np.int64)
X = df.drop(columns=["Churn", "CustomerID"])

In [ ]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

In [ ]:
# подготовим данные
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# все вместе в препроцессор
preprocessor = ColumnTransformer([
    ("num", num_pipe, numeric_cols),
    ("cat", cat_pipe, categorical_cols)
])

In [ ]:

run = wandb.init(
    entity = 'martyanov-an-',
    project = 'GP_5',
    name = 'test_2',
    config = {
        'dataset': 'train',
        'categorial_imputer': 'SimpleImputer',
        'numeric_imputer': 'SimpleImputer',
        'encoder': 'OrdinalEncoder'

    }
    )
run.finish()

In [ ]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train = preprocessor.fit_transform(X_train_df)
X_test = preprocessor.transform(X_test_df)

In [ ]:


X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
cat_idxs = list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols)))
enc = preprocessor.named_transformers_["cat"].named_steps["encoder"]

cat_dims = [len(c) + 1 for c in enc.categories_]

# фиксим индексы для эмбеддингов
def prepare_data(X):
    tmp = X.copy()
    tmp[:, cat_idxs] += 1
    return tmp

X_train = prepare_data(X_train)
X_test = prepare_data(X_test)

print(X_train.shape)
print(cat_idxs)
print(cat_dims)

X_train shape: (40837, 56)
cat_idxs: [34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]
cat_dims: [731, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 17, 3, 8, 5, 9, 4]


In [ ]:
import torch

# параметры для табнета
tab_params = {
    "n_d": 16,
    "n_a": 16,
    "n_steps": 3,
    "gamma": 1.3,
    "lambda_sparse": 1e-3,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": dict(lr=0.01),
    "mask_type": "sparsemax",
    "seed": 42
}

clf_tab = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=4,
    **tab_params
)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [ ]:
run = wandb.init(
    project='GP_5',
    name='tabnet_final',
    config={'batch': 1024}
)

# запускаем обучение
clf_tab.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_test, y_test)],
    eval_name=['test'],
    eval_metric=['auc'],
    max_epochs=20,
    patience=5,
    batch_size=1024,
    virtual_batch_size=128
)

final_auc = roc_auc_score(y_test, clf_tab.predict_proba(X_test)[:, 1])
wandb.log({'auc': final_auc})
print(f"Result AUC: {final_auc}")
run.finish()

epoch 0  | loss: 0.67509 | train_auc: 0.5049  | valid_auc: 0.50691 |  0:00:12s
epoch 1  | loss: 0.60584 | train_auc: 0.5044  | valid_auc: 0.50137 |  0:00:21s
epoch 2  | loss: 0.60115 | train_auc: 0.54091 | valid_auc: 0.54299 |  0:00:27s
epoch 3  | loss: 0.59676 | train_auc: 0.56789 | valid_auc: 0.5603  |  0:00:31s
epoch 4  | loss: 0.59442 | train_auc: 0.58542 | valid_auc: 0.57407 |  0:00:36s
epoch 5  | loss: 0.59313 | train_auc: 0.58919 | valid_auc: 0.58735 |  0:00:41s
epoch 6  | loss: 0.59098 | train_auc: 0.59818 | valid_auc: 0.59401 |  0:00:46s
epoch 7  | loss: 0.58976 | train_auc: 0.59815 | valid_auc: 0.5937  |  0:00:52s
epoch 8  | loss: 0.58838 | train_auc: 0.61242 | valid_auc: 0.6074  |  0:00:57s
epoch 9  | loss: 0.58677 | train_auc: 0.60993 | valid_auc: 0.60542 |  0:01:02s
epoch 10 | loss: 0.58698 | train_auc: 0.60598 | valid_auc: 0.60024 |  0:01:08s
epoch 11 | loss: 0.58657 | train_auc: 0.60994 | valid_auc: 0.60362 |  0:01:12s
epoch 12 | loss: 0.58601 | train_auc: 0.61884 | vali

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Final Test AUC: 0.6190058335674817


final_test_auc,▁
final_test_auc,0.61901


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# кросс-слой
class Cross(nn.Module):
    def __init__(self, dim, n):
        super().__init__()
        self.n = n
        self.ws = nn.ParameterList([nn.Parameter(torch.randn(dim, 1)) for _ in range(n)])
        self.bs = nn.ParameterList([nn.Parameter(torch.randn(dim, 1)) for _ in range(n)])

    def forward(self, x):
        x0 = x.unsqueeze(2)
        curr = x0
        for i in range(self.n):
            # считаем по формуле из статьи
            xw = torch.matmul(curr.transpose(1, 2), self.ws[i])
            curr = torch.matmul(x0, xw) + self.bs[i] + curr
        return curr.squeeze(2)

# общая модель dcn
class DCNModel(nn.Module):
    def __init__(self, d, n_cross=3, h_dims=[256, 128], p=0.2):
        super().__init__()
        self.cross = Cross(d, n_cross)

        # тут обычные полносвязные слои
        layers = []
        curr = d
        for h in h_dims:
            layers.extend([nn.Linear(curr, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p)])
            curr = h
        self.dense = nn.Sequential(*layers)
        self.final = nn.Linear(d + h_dims[-1], 1)

    def forward(self, x):
        out_c = self.cross(x)
        out_d = self.dense(x)
        # склеиваем выходы
        res = torch.cat([out_c, out_d], dim=1)
        return torch.sigmoid(self.final(res))

DCN Test AUC: 0.4952
Precision: 0.2844
Recall: 0.5201

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.47      0.57      7268
           1       0.28      0.52      0.37      2942

    accuracy                           0.48     10210
   macro avg       0.50      0.50      0.47     10210
weighted avg       0.59      0.48      0.51     10210



final_test_auc,▁
precision,▁
recall,▁
final_test_auc,0.49524
precision,0.28444
recall,0.52005
